# Tokenization experiments

In [ ]:
from miditok import REMI, TokenizerConfig
from miditoolkit import MidiFile
from pretty_midi import PrettyMIDI
from pretty_midi.utilities import program_to_instrument_name
import pandas as pd
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from utils import iter_midi_paths
from random import sample

midi_file_paths = iter_midi_paths(Path("data/lmd_matched"))  # Update this path
config = TokenizerConfig(
    num_velocities=32,
    beat_res={(0, 2): 8, (2, 4): 4},  
    use_chords=True,
    use_time_signatures=True,
)
tok = REMI(config)


rows = []
sample_paths = sample(list(midi_file_paths), 500)

for p in tqdm(sample_paths):
    try:
        midi_miditools = MidiFile(p)
        midi_pretty = PrettyMIDI(str(p))
        tokens = tok(midi_miditools)

        rows.append({
            "path": str(p),
            "num_tokens": len(tokens),
            "tokens": tokens,                                    # list -> stored as object
            "instruments": list({program_to_instrument_name(inst.program) for inst in midi_miditools.instruments}),
            "num_instruments": len(midi_miditools.instruments),
            "duration": float(midi_pretty.get_end_time()),
            "tempo_changes": tuple(map(list, midi_pretty.get_tempo_changes())),  # (times, tempi) -> lists
            "time_signatures": [str(ts) for ts in midi_pretty.time_signature_changes],
            "key_signatures": [str(ks) for ks in midi_pretty.key_signature_changes],
        })
    except Exception as e:
        # at least keep track of failures
        rows.append({"path": str(p), "error": repr(e)})

df = pd.DataFrame.from_records(rows)
df.describe()


In [ ]:
df

In [ ]:
df["instruments"].explode().value_counts(normalize=True)[:10].plot(kind="bar", figsize=(12,6))
plt.title("Instrument Distribution in Sampled MIDI Files")
plt.xlabel("Instruments")

In [ ]:
df[df["duration"]== df["duration"].max()]

In [ ]:
for path in df[df["duration"]== df["duration"].max()]["path"].values:
    print(path)

In [ ]:
from IPython.display import Audio
output_audio = PrettyMIDI(str(Path("data/lmd_matched/H/M/I/TRHMIJC12903CAD69B/3015fd6ea6a771e9fe9cda0aac752e13.mid"))).synthesize()
Audio(output_audio, rate=44100)

In [ ]:
len(PrettyMIDI(str(Path("data/lmd_matched/H/M/I/TRHMIJC12903CAD69B/3015fd6ea6a771e9fe9cda0aac752e13.mid"))).instruments)

In [ ]:
from miditok import REMI
config = TokenizerConfig(
    num_velocities=32,
    beat_res={(0, 2): 8, (2, 4): 4},  
    use_chords=True,
    use_time_signatures=True
)
tokenizer = REMI(config)

toks= tokenizer("data/lmd_matched/H/M/I/TRHMIJC12903CAD69B/3015fd6ea6a771e9fe9cda0aac752e13.mid")
len(toks[0].ids)

In [ ]:
# === Auto-tune Miditok REMI with a MIDI distance metric (Jupyter-ready) ===
# Metrics include: tokens_per_second, p95 token length, note_loss_rate,
# tempo/timesig diffs, error_rate, and a new framewise Jaccard distance.

from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
import json, math, statistics, traceback, os

import numpy as np
import pandas as pd

from tqdm import tqdm_notebook as tqdm


from miditok import REMI, TokenizerConfig
from miditoolkit import MidiFile


# ---------- Tempo-aware time conversion helpers ----------
def _tempo_segments(midi: MidiFile):
    """Return segments [(tick_start, tick_end, bpm)] covering from 0..end."""
    tpb = max(1, midi.ticks_per_beat)
    end_tick = max(
        [(n.end) for inst in midi.instruments for n in inst.notes]
        + [midi.max_tick]
        + [0]
    )
    tempos = sorted(midi.tempo_changes, key=lambda t: t.time)
    if not tempos:
        # default 120 bpm if none present
        class T:
            pass

        T.time = 0
        T.tempo = 120.0
        tempos = [T]
    segs = []
    for i, t in enumerate(tempos):
        t0 = int(t.time)
        bpm = float(t.tempo)
        t1 = int(tempos[i + 1].time) if i + 1 < len(tempos) else int(end_tick)
        if t1 > t0:
            segs.append((t0, t1, bpm))
    if segs and segs[-1][1] < end_tick:
        # pad to end with last tempo
        s0, _, bpm = segs[-1]
        segs.append((segs[-1][1], int(end_tick), bpm))
    return segs, tpb


def build_tick_to_sec(midi: MidiFile):
    """Return a function tick->seconds using the MIDI's tempo map."""
    segs, tpb = _tempo_segments(midi)
    # Precompute cumulative seconds
    cum = []
    acc = 0.0
    for a, b, bpm in segs:
        beats = (b - a) / tpb
        dur = (60.0 / max(1e-6, bpm)) * beats
        cum.append((a, b, bpm, acc))
        acc += dur

    def tick_to_sec(tick: int) -> float:
        for a, b, bpm, acc0 in cum:
            if a <= tick < b:
                beats = (tick - a) / tpb
                return acc0 + (60.0 / max(1e-6, bpm)) * beats
        # after last segment
        a, b, bpm, acc0 = cum[-1]
        beats = (tick - b) / tpb
        return acc0 + (60.0 / max(1e-6, bpm)) * beats

    total_seconds = acc
    return tick_to_sec, float(total_seconds)


# ---------- Framewise Jaccard distance ----------
def midi_to_active_frames(
    midi: MidiFile, fps: float = 50.0, pitch_low: int = 21, pitch_high: int = 108
):
    """
    Map MIDI to a set of active (frame_index, pitch) pairs at fps frames/sec.
    Range defaults to piano A0(21)..C8(108); adjust for your data if needed.
    """
    tick_to_sec, total_sec = build_tick_to_sec(midi)
    S = set()
    for inst in midi.instruments:
        for note in inst.notes:
            p = int(note.pitch)
            if p < pitch_low or p > pitch_high:
                continue
            t0 = max(0.0, tick_to_sec(int(note.start)))
            t1 = max(t0, tick_to_sec(int(note.end)))
            if t1 <= t0:
                continue
            f0 = int(math.floor(t0 * fps))
            f1 = int(math.floor(t1 * fps))
            # mark active frames [f0, f1)
            for f in range(f0, max(f0 + 1, f1)):
                S.add((f, p))
    return S, int(math.ceil(total_sec * fps))


def jaccard_distance_frames(m1: MidiFile, m2: MidiFile, fps: float = 50.0) -> float:
    """
    1 - Jaccard index between sets of active (frame, pitch). Lower = more similar.
    """
    S1, _ = midi_to_active_frames(m1, fps=fps)
    S2, _ = midi_to_active_frames(m2, fps=fps)
    if not S1 and not S2:
        return 0.0
    inter = len(S1.intersection(S2))
    union = len(S1.union(S2))
    return 1.0 - (inter / union if union else 1.0)


# ---------- Other metrics ----------
def read_midi_duration_seconds(midi: MidiFile) -> float:
    _, total = build_tick_to_sec(midi)
    return total if total > 0 else 1e-6


def count_notes(midi: MidiFile) -> int:
    return sum(len(inst.notes) for inst in midi.instruments)


def count_tempos(midi: MidiFile) -> int:
    return len(midi.tempo_changes)


def count_timesigs(midi: MidiFile) -> int:
    return len(midi.time_signature_changes)


# ---------- Configs & tokenization ----------
def default_grid() -> List[Dict[str, Any]]:
    """Compact but diverse REMI TokenizerConfig grid."""
    grids = []
    beat_res_opts = [
        {(0, 4): 8, (4, 12): 4},
        {(0, 4): 4, (4, 12): 4},
        {(0, 4): 12, (4, 12): 6},
    ]
    nb_vels = [16, 32, 64]
    chords = [True, False]
    rests = [True, False]
    tempos = [True]
    timesigs = [True]
    programs = [True]
    for br in beat_res_opts:
        for nv in nb_vels:
            for uc in chords:
                for ur in rests:
                    for ut in tempos:
                        for uts in timesigs:
                            for up in programs:
                                grids.append(
                                    dict(
                                        beat_res=br,
                                        nb_velocities=nv,
                                        use_chords=uc,
                                        use_rests=ur,
                                        use_tempos=ut,
                                        use_time_signatures=uts,
                                        use_programs=up,
                                    )
                                )
    return grids


def make_tokenizer(cfg_dict: Dict[str, Any]) -> REMI:
    config = TokenizerConfig(**cfg_dict)
    return REMI(config)


def tokenize_ids(tokenizer: REMI, midi: MidiFile) -> List[int]:
    toks = tokenizer(midi)
    ids = getattr(toks, "ids", None) or getattr(toks, "ids_list", None)
    if ids is None:
        if isinstance(toks, list) and all(isinstance(x, int) for x in toks):
            ids = toks
        else:
            raise RuntimeError("TokSequence had no .ids/.ids_list")
    return list(ids)


def decode_ids(tokenizer: REMI, ids: List[int]) -> MidiFile:
    return tokenizer.decode(ids)


# ---------- Ranking ----------
def zscore(arr: np.ndarray) -> np.ndarray:
    m = np.nanmean(arr)
    s = np.nanstd(arr)
    return (arr - m) / (s if s > 1e-12 else 1.0)


def rank_configs(df: pd.DataFrame, weights: Dict[str, float]) -> pd.DataFrame:
    comp = {}
    comp["tokens_per_second"] = zscore(df["tokens_per_second"].to_numpy())
    comp["p95_tokens_per_file"] = zscore(df["p95_tokens_per_file"].to_numpy())
    comp["note_loss_rate"] = zscore(df["note_loss_rate"].to_numpy())
    comp["struct_diff"] = zscore((df["tempo_diff"] + df["timesig_diff"]).to_numpy())
    comp["error_rate"] = zscore(df["error_rate"].to_numpy())
    comp["dist_jaccard"] = zscore(df["dist_jaccard_frames"].to_numpy())

    score = (
        weights.get("w_len", 1.0) * comp["tokens_per_second"]
        + weights.get("w_len_p95", 0.5) * comp["p95_tokens_per_file"]
        + weights.get("w_loss", 2.0) * comp["note_loss_rate"]
        + weights.get("w_struct", 1.0) * comp["struct_diff"]
        + weights.get("w_err", 3.0) * comp["error_rate"]
        + weights.get("w_dist", 2.0) * comp["dist_jaccard"]
    )
    out = df.copy()
    out["score"] = score
    return out.sort_values("score", ascending=True)


# ---------- Main notebook runner ----------
# >>> EDIT THESE <<<
INPUT_DIR = Path("data/lmd_matched")  # <= change me
OUT_DIR = Path("./out_autotune")  # where to save CSVs
MAX_FILES = 200  # evaluate at most N files
FPS_FOR_DISTANCE = 50.0  # frames/sec for Jaccard distance
WEIGHTS = {
    "w_len": 1.0,
    "w_len_p95": 0.5,
    "w_loss": 2.0,
    "w_struct": 1.0,
    "w_err": 3.0,
    "w_dist": 2.0,
}
GRID_JSON = True  # e.g., Path("my_grid.json") to override the grid
SKIP_DECODE = False  # set True to skip decode & distance (faster; distance will be NaN)


# Collect files
assert INPUT_DIR.exists(), f"{INPUT_DIR} does not exist"
exts = {".mid", ".midi"}
files = sorted([p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in exts])
if MAX_FILES:
    files = files[:MAX_FILES]
assert files, "No MIDI files found."

# Load MIDIs
dataset_midis: List[Tuple[Path, MidiFile]] = []
it = tqdm(files, desc="Loading MIDIs") 
for f in it:
    try:
        dataset_midis.append((f, MidiFile(str(f))))
    except Exception:
        print(f"[WARN] Failed to load {f}:\n{traceback.format_exc()}")

assert dataset_midis, "No valid MIDI could be loaded."

# Precompute original stats
orig_stats = []
for _, m in dataset_midis:
    try:
        dur = read_midi_duration_seconds(m)
    except Exception:
        # fallback: rough estimate with 120 bpm
        dur = (m.max_tick / max(1, m.ticks_per_beat)) * 0.5
    orig_stats.append(
        dict(
            notes=count_notes(m),
            tempos=count_tempos(m),
            timesigs=count_timesigs(m),
            seconds=max(1e-6, dur),
        )
    )

# Build grid
if GRID_JSON:
    cfg_list = json.loads(Path(GRID_JSON).read_text(encoding="utf-8"))
else:
    cfg_list = default_grid()

rows = []
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg_iter = tqdm(cfg_list, desc="Configs") 
for cfg in cfg_iter:
    # Per-config aggregations
    tok_lens = []
    errs = 0
    note_losses = []
    tempo_diffs = []
    timesig_diffs = []
    jaccard_dists = []

    try:
        tokenizer = make_tokenizer(cfg)
    except Exception:
        print(f"[WARN] Bad TokenizerConfig: {cfg}\n{traceback.format_exc()}")
        rows.append(
            dict(
                config=json.dumps(cfg),
                error_rate=1.0,
                tokens_per_second=np.nan,
                median_tokens_per_file=np.nan,
                p95_tokens_per_file=np.nan,
                note_loss_rate=np.nan,
                tempo_diff=np.nan,
                timesig_diff=np.nan,
                dist_jaccard_frames=np.nan,
            )
        )
        continue

    for (path, midi), orig in zip(dataset_midis, orig_stats):
        try:
            ids = tokenize_ids(tokenizer, midi)
            tok_lens.append(len(ids))
            if not SKIP_DECODE:
                try:
                    dec = decode_ids(tokenizer, ids)
                    # Note loss (relative)
                    n_loss = abs(count_notes(dec) - orig["notes"]) / max(
                        1, orig["notes"]
                    )
                    t_diff = abs(count_tempos(dec) - orig["tempos"])
                    s_diff = abs(count_timesigs(dec) - orig["timesigs"])
                    # NEW: framewise Jaccard distance
                    d_jacc = jaccard_distance_frames(midi, dec, fps=FPS_FOR_DISTANCE)
                except Exception:
                    # decoding failed => penalize
                    n_loss, t_diff, s_diff, d_jacc = 1.0, 5.0, 5.0, 1.0
                note_losses.append(n_loss)
                tempo_diffs.append(t_diff)
                timesig_diffs.append(s_diff)
                jaccard_dists.append(d_jacc)

        except Exception:
            errs += 1  # count error, move on

    n_ok = len(dataset_midis) - errs
    if n_ok == 0:
        tps = med = p95 = np.nan
    else:
        total_seconds = float(sum(o["seconds"] for o in orig_stats))
        total_tokens = float(sum(tok_lens))
        tps = total_tokens / max(1e-9, total_seconds)
        med = float(statistics.median(tok_lens))
        p95 = float(np.percentile(tok_lens, 95))

    rows.append(
        dict(
            config=json.dumps(cfg),
            error_rate=errs / len(dataset_midis),
            tokens_per_second=tps,
            median_tokens_per_file=med,
            p95_tokens_per_file=p95,
            note_loss_rate=float(
                np.nan if SKIP_DECODE or not note_losses else np.mean(note_losses)
            ),
            tempo_diff=float(
                np.nan if SKIP_DECODE or not tempo_diffs else np.mean(tempo_diffs)
            ),
            timesig_diff=float(
                np.nan if SKIP_DECODE or not timesig_diffs else np.mean(timesig_diffs)
            ),
            dist_jaccard_frames=float(
                np.nan if SKIP_DECODE or not jaccard_dists else np.mean(jaccard_dists)
            ),
        )
    )

# Results dataframe
df = pd.DataFrame(rows)
ranked = rank_configs(df, WEIGHTS)

# Save artifacts
(df).to_csv(OUT_DIR / "results.csv", index=False)
(ranked).to_csv(OUT_DIR / "results_ranked.csv", index=False)

# Save best config
if len(ranked) > 0:
    best_cfg = json.loads(ranked.iloc[0]["config"])
    with open(OUT_DIR / "best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_cfg, f, ensure_ascii=False, indent=2)

# Show top rows
print("Top 5 configs:")
display(ranked.head(5))

print("\nBest TokenizerConfig dict:")
if len(ranked) > 0:
    display(best_cfg)
else:
    print("No valid configs.")

In [ ]:
TOKENIZER_PARAMS = {
    "pitch_range": (21, 109),
    "beat_res": {(0, 2): 8, (2, 4): 4},
    "num_velocities": 32,
    "special_tokens": ["PAD", "BOS", "EOS", "MASK"],
    "use_chords": True,
    "use_rests": False,
    "use_tempos": True,
    "use_time_signatures": False,
    "use_programs": False,
    "num_tempos": 32,  # number of tempo bins
    "tempo_range": (40, 250),  # (min, max)
}
config = TokenizerConfig(**TOKENIZER_PARAMS)

# Creates the tokenizer
tok = REMI(config)
tok(tokens).dump_midi(Path("decoded_midi.mid"))

In [ ]:
from IPython.display import Audio
output_audio = pretty_midi.PrettyMIDI("decoded_midi.mid").synthesize(fs=44100)
Audio(output_audio, rate=44100)

In [ ]:
output_audio = pretty_midi.PrettyMIDI(str(sample[299])).synthesize(fs=44100)
Audio(output_audio, rate=44100)